<a href="https://colab.research.google.com/github/avinashnikhil/Fleetanalytics/blob/main/Fleet_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# CELL A1: Imports and settings
# This cell installs necessary libraries, imports modules, creates project directories,
# and defines all configurations, settings, and the master random generator for our simulation.

# Step 1: Install duckdb and plotly quietly
!pip install -q duckdb plotly

# Step 2: Import all required libraries
import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
from pathlib import Path

# Step 3: Create target directories for our files
# We create 'data' and 'sql' folders so we can store CSV files and SQL queries later
Path("data").mkdir(parents=True, exist_ok=True)
Path("sql").mkdir(parents=True, exist_ok=True)

# Step 4: Define all project settings as CAPITAL_LETTER variables

# === General Settings ===
SEED = 42  # Seed for the random number generator to guarantee reproducible results
NUMBER_OF_ROBOTS = 50  # Total number of autonomous robots in our simulated fleet
NUMBER_OF_DAYS = 90  # The length of our historical simulation in calendar days
START_DATE = "2026-07-01"  # The start date of our fleet operation records

# === Demand Settings ===
AVERAGE_MISSIONS_PER_ROBOT_PER_DAY = 9  # Baseline target of daily trips for each active robot
WEEKEND_DEMAND_MULTIPLIER = 1.3  # Increase in target mission demand during Saturdays and Sundays
RAIN_PROBABILITY = 0.25  # Daily probability that weather is rainy instead of clear
FIRST_HOUR = 8  # Earliest hour of the day a delivery mission can begin (8 AM)
LAST_HOUR = 21  # Latest hour of the day a delivery mission can begin (9 PM)
RUSH_HOURS = [11, 12, 13, 17, 18, 19]  # Lunch and dinner peak delivery timeframes
RUSH_HOUR_DEMAND_WEIGHT = 2.5  # Relative weight of demand inside rush hours versus normal hours

# === Zones Settings ===
ZONE_SHARES = {
    "Campus": 0.25,
    "Downtown": 0.25,
    "Riverside": 0.20,
    "Suburb": 0.18,
    "Old Town": 0.12
}  # Geographic market distribution shares of all starting missions

# === Trip Length Settings ===
MEDIAN_DISTANCE_KM = 1.5  # Median travel distance for normal log-normal distribution of trips
DISTANCE_SPREAD = 0.5  # Sigma/spread value for the distance log-normal distribution
MINIMUM_DISTANCE_KM = 0.3  # Absolute minimum distance permitted for any single mission
MAXIMUM_DISTANCE_KM = 6.0  # Absolute maximum distance permitted for any single mission

# === Software Versions Settings ===
FIRST_V22_DAY = 31  # Simulation day on which fleet starts running version v2.2
FIRST_V23_DAY = 61  # Simulation day on which even-numbered robots receive version v2.3

# === Interventions Settings ===
BASE_INTERVENTIONS_PER_KM = 0.03  # Baseline expected frequency of human remote assistance events per kilometer
VERSION_INTERVENTION_MULTIPLIER = {
    "v2.1": 1.00,
    "v2.2": 0.85,
    "v2.3": 1.50
}  # Impact of firmware software versions on the baseline intervention rate
RAIN_INTERVENTION_MULTIPLIER = 1.6  # Multiplier applied to interventions when weather is rainy
V23_IN_RAIN_EXTRA_MULTIPLIER = 1.1  # Special penalty factor for v2.3 code running in wet conditions
OLD_TOWN_INTERVENTION_MULTIPLIER = 2.0  # Multiplier for old cobblestone roads and dense alleys
NIGHT_INTERVENTION_MULTIPLIER = 1.3  # Multiplier representing low visibility conditions during dark hours
NIGHT_START_HOUR = 20  # Hour of day (8 PM) representing start of nighttime visibility conditions

# === Intervention Reasons Settings ===
REASON_BASE_WEIGHTS = {
    "crosswalk_confusion": 25,
    "blocked_path": 25,
    "low_confidence": 20,
    "sensor_obstructed": 10,
    "localization_lost": 10,
    "curb_or_door": 10
}  # Base relative probability weight distribution for telemetry issue categories
V23_CROSSWALK_WEIGHT_MULTIPLIER = 3  # Multiplication factor for crosswalk issues under v2.3 software
RAIN_SENSOR_WEIGHT_MULTIPLIER = 3  # Multiplication factor for sensor blockages when rain is present
OLD_TOWN_BLOCKED_AND_CURB_WEIGHT_MULTIPLIER = 2  # Multiplication factor for physical obstacles in Old Town
INTERVENTION_MEDIAN_SECONDS = 45  # Median time of a remote operator recovery action
INTERVENTION_DURATION_SPREAD = 0.6  # Log-normal distribution shape parameter for intervention duration
MIN_INTERVENTION_SECONDS = 10  # Minimum allowed duration for physical intervention records
MAX_INTERVENTION_SECONDS = 600  # Maximum allowed duration for physical intervention records

# === Speed and Timing Settings ===
PLANNED_SPEED_KMH = 5.5  # Baseline speed used to calculate ideal promised travel ETA
ACTUAL_SPEED_KMH = 5.0  # Baseline physical speed of robot driving mechanics
SPEED_RANDOM_RANGE = (0.9, 1.1)  # Multiplication bounds of random speed variation factor
PROMISED_BUFFER_MINUTES = 5  # Generous minute cushion added to promised arrival time target
RAIN_SLOWDOWN = 1.2  # Time slowdown multiplier (increases duration) under rain conditions
OLD_TOWN_SLOWDOWN = 1.4  # Time slowdown multiplier (increases duration) inside Old Town
RUSH_HOUR_SLOWDOWN = 1.15  # Time slowdown multiplier (increases duration) during peak traffic

# === Battery Settings ===
HEALTHY_BATTERY_HEALTH_RANGE = (0.85, 1.00)  # Random uniform range for healthy robot batteries
TIRED_BATTERY_ROBOT_IDS = ["R007", "R019", "R023", "R031", "R044"]  # Specific robots with degraded batteries
TIRED_BATTERY_HEALTH = 0.65  # Battery capacity coefficient for specified degraded robots
BATTERY_DRAIN_PERCENT_PER_KM = 5.0  # Expected baseline battery drainage per kilometer traveled
START_CHARGE_RANGE = (25, 100)  # Initial state of charge percentage boundaries before mission starts
BATTERY_FAILURE_THRESHOLD_PCT = 5  # Remaining charge limit below which a critical battery failure occurs

# === Failures Settings ===
RANDOM_FAILURE_RATE = 0.015  # Probability of experiencing an unresolvable hardware/system trip failure
RANDOM_FAILURE_REASONS = ["stuck_or_blocked", "hardware_fault"]  # Categories of critical failure codes

# === Dirty Data Settings ===
NUMBER_OF_DUPLICATE_MISSIONS = 20  # Count of identical mission records to insert for testing data cleaning
NUMBER_OF_NEGATIVE_DURATIONS = 30  # Count of physical logs corrupted with negative run durations
NUMBER_OF_ZERO_DISTANCES = 25  # Count of mission records corrupted with zeroed distance traveled

# === Cost Settings ===
COST_ASSUMPTIONS = {
    "robot_depreciation_per_day": 7.00,
    "maintenance_per_km": 0.10,
    "energy_per_km": 0.01,
    "operator_per_minute": 0.60,
    "handoff_per_intervention": 1.50,
    "overhead_per_mission": 0.40
}  # Operational financial model parameters for calculating fleet cost metrics

# === Paths Settings ===
DATA_FOLDER = "data"  # Subdirectory path target for data outputs
SQL_FOLDER = "sql"  # Subdirectory path target for queries
DATABASE_FILE = "fleet.duckdb"  # Persistent storage filename of DuckDB metrics layer

# Step 5: Initialize the master random number generator
random_generator = np.random.default_rng(SEED)

# Step 6: Print checks to verify the configurations
# Calculate total share weight to show it equals 1.0
total_zone_shares = sum(ZONE_SHARES.values())
start_date_parsed = pd.to_datetime(START_DATE)
end_date_parsed = start_date_parsed + pd.Timedelta(days=NUMBER_OF_DAYS - 1)

print("=== CONFIGURATION SETTINGS VERIFICATION ===")
print(f"Number of Robots: {NUMBER_OF_ROBOTS}")
print(f"Number of Days: {NUMBER_OF_DAYS}")
print(f"First Date of Simulation: {start_date_parsed.strftime('%Y-%m-%d')}")
print(f"Last Date of Simulation: {end_date_parsed.strftime('%Y-%m-%d')}")
print(f"Zone Share Distribution: {ZONE_SHARES}")
print(f"Total Zone Shares Sum Check: {total_zone_shares:.2f}")

# === What this cell does ===
# 1. Silently installs and imports all crucial packages like pandas, numpy, duckdb, and plotly.
# 2. Configures directories 'data' and 'sql' for future storage needs and sets up a reproducible NumPy generator.
# 3. Establishes a standard environment containing all of our operational project constants in one easy-to-manage file section.

# === You should see ===
# A verification printout listing 50 robots, 90 days, starting date 2026-07-01, ending date 2026-09-28,
# the exact zone share dictionary, and a confirmation sum check of exactly 1.00.

=== CONFIGURATION SETTINGS VERIFICATION ===
Number of Robots: 50
Number of Days: 90
First Date of Simulation: 2026-07-01
Last Date of Simulation: 2026-09-28
Zone Share Distribution: {'Campus': 0.25, 'Downtown': 0.25, 'Riverside': 0.2, 'Suburb': 0.18, 'Old Town': 0.12}
Total Zone Shares Sum Check: 1.00


In [4]:
# CELL B1: Create robots table
# This cell generates the master fleet list of delivery robots, establishes their purchase history,
# and models their operational battery health parameters using random numbers.

# Step 1: Generate the sequential list of zero-padded robot IDs
# We use the NUMBER_OF_ROBOTS setting to create a range from 1 to 50, padded to three digits with a leading 'R'
robot_id_list = [
    f"R{index:03d}"
    for index in range(1, NUMBER_OF_ROBOTS + 1)
]

# Step 2: Generate random battery health values for healthy robots
# We draw uniform floating-point values between the configured min and max, then round to 2 decimals
healthy_min_health = HEALTHY_BATTERY_HEALTH_RANGE[0]
healthy_max_health = HEALTHY_BATTERY_HEALTH_RANGE[1]

battery_health_values = random_generator.uniform(
    low=healthy_min_health,
    high=healthy_max_health,
    size=NUMBER_OF_ROBOTS
)
battery_health_values = np.round(battery_health_values, decimals=2)

# Step 3: Initialize the robots DataFrame
robots_df = pd.DataFrame({
    "robot_id": robot_id_list,
    "battery_health": battery_health_values
})

# Step 4: Overwrite battery health values for the tired robots listed in the settings
# We use the .isin() method to find the matching robot IDs and replace their health with the tired constant
robots_df.loc[
    robots_df["robot_id"].isin(TIRED_BATTERY_ROBOT_IDS),
    "battery_health"
] = TIRED_BATTERY_HEALTH

# Step 5: Generate random purchase dates within the 365 days prior to START_DATE
# We calculate the start date anchor and generate a random integer number of days to subtract from it
start_date_parsed = pd.to_datetime(START_DATE)
random_days_prior = random_generator.integers(
    low=0,
    high=365,
    size=NUMBER_OF_ROBOTS
)

# We subtract the random day offset from our project start date to find the historical purchase date
purchase_dates = start_date_parsed - pd.to_timedelta(random_days_prior, unit="D")

# FIX: Since purchase_dates is a DatetimeIndex, we access the .date attribute directly
robots_df["purchase_date"] = purchase_dates.date

# Step 6: Print checks to verify the DataFrame structure and contents
print("=== ROBOTS DATAFRAME FIRST 10 ROWS ===")
display(robots_df.head(10))

print("\n=== DATAFRAME SHAPE ===")
print(robots_df.shape)

print("\n=== BATTERY HEALTH SUMMARY STATISTICS ===")
display(robots_df["battery_health"].describe())

print("\n=== TIRED ROBOTS VERIFICATION ===")
tired_robots_check = robots_df[robots_df["robot_id"].isin(TIRED_BATTERY_ROBOT_IDS)]
display(tired_robots_check)

print("\n=== COUNT OF ROBOTS WITH HEALTH BELOW 0.70 ===")
degraded_count = (robots_df["battery_health"] < 0.70).sum()
print(f"Number of robots with degraded battery health (< 0.70): {degraded_count}")

# === What this cell does ===
# 1. Generates a base list of 50 unique, zero-padded robot ID strings from 'R001' to 'R050'.
# 2. Applies randomized battery health values ranging from 0.85 to 1.00, manually downgrading five targeted robot IDs to a weary 0.65 health score.
# 3. Retroactively assigns purchase dates to every robot falling randomly within the year prior to the fleet operational launch date.

# === You should see ===
# 1. A preview of the first 10 robots with varying purchase dates in 2025/2026 and battery health scores.
# 2. A DataFrame shape of exactly (50, 3) representing 50 rows and 3 defined columns.
# 3. A descriptive stats summary showing a minimum battery health of exactly 0.65 and maximum of 1.00.
# 4. A filtered view of exactly 5 robots (R007, R019, R023, R031, R044) showing battery_health of exactly 0.65.
# 5. A final count confirming exactly 5 robots have a battery health reading below 0.70.

=== ROBOTS DATAFRAME FIRST 10 ROWS ===


,robot_id,battery_health,purchase_date
0,R001,0.89,2026-04-27
1,R002,0.89,2026-01-02
2,R003,0.95,2026-04-24
3,R004,0.93,2026-03-03
4,R005,0.97,2025-11-02
5,R006,0.95,2026-05-10
6,R007,0.65,2025-10-27
7,R008,0.97,2026-05-25
8,R009,0.88,2026-04-29
9,R010,0.85,2025-11-29



=== DATAFRAME SHAPE ===
(50, 3)

=== BATTERY HEALTH SUMMARY STATISTICS ===


,battery_health
count,50.000000
mean,0.895000
std,0.091813
min,0.650000
25%,0.872500
50%,0.920000
75%,0.950000
max,1.000000



=== TIRED ROBOTS VERIFICATION ===


,robot_id,battery_health,purchase_date
6,R007,0.65,2025-10-27
18,R019,0.65,2025-11-20
22,R023,0.65,2026-05-07
30,R031,0.65,2026-03-23
43,R044,0.65,2026-03-02



=== COUNT OF ROBOTS WITH HEALTH BELOW 0.70 ===
Number of robots with degraded battery health (< 0.70): 5


In [5]:
# CELL C1: Create days table
# This cell constructs an operational calendar dataframe containing sequential day indices,
# corresponding calendar dates, weekend flags, demand scaling multipliers, and randomized weather states.

# Step 1: Create a sequential array of day numbers from 1 up to NUMBER_OF_DAYS
day_numbers_array = np.arange(1, NUMBER_OF_DAYS + 1)

# Step 2: Generate sequential calendar dates starting from START_DATE
# We convert START_DATE to a Timestamp and add a range of timedeltas corresponding to our day counts
start_timestamp = pd.to_datetime(START_DATE)
calendar_dates = start_timestamp + pd.to_timedelta(day_numbers_array - 1, unit="D")

# Step 3: Identify weekend status for every date
# We extract the day of the week where Saturday (5) and Sunday (6) represent weekends
day_of_week_indices = calendar_dates.dayofweek
is_weekend_flags = (day_of_week_indices == 5) | (day_of_week_indices == 6)

# Step 4: Determine demand multipliers based on weekend status
# We assign WEEKEND_DEMAND_MULTIPLIER (1.3) on weekends and 1.0 on weekdays using np.where
demand_multipliers = np.where(
    is_weekend_flags,
    WEEKEND_DEMAND_MULTIPLIER,
    1.0
)

# Step 5: Simulate weather states with one random draw per day
# We use our master random_generator to decide weather using the configured RAIN_PROBABILITY
weather_random_draws = random_generator.random(size=NUMBER_OF_DAYS)
weather_states = np.where(
    weather_random_draws < RAIN_PROBABILITY,
    "rain",
    "clear"
)

# Step 6: Package these arrays into the final days_df DataFrame
days_df = pd.DataFrame({
    "day_number": day_numbers_array,
    "date": calendar_dates.date,
    "is_weekend": is_weekend_flags,
    "weather": weather_states,
    "demand_multiplier": demand_multipliers
})

# Step 7: Print diagnostic verification checks
print("=== DAYS DATAFRAME FIRST 10 ROWS ===")
display(days_df.head(10))

print("\n=== DATAFRAME SHAPE ===")
print(days_df.shape)

print("\n=== WEATHER COUNTS ===")
display(days_df["weather"].value_counts())

print("\n=== WEEKEND COUNTS ===")
display(days_df["is_weekend"].value_counts())

print("\n=== MIN AND MAX DATES ===")
min_date_val = days_df["date"].min()
max_date_val = days_df["date"].max()
print(f"First simulated date: {min_date_val}")
print(f"Last simulated date: {max_date_val}")

# === What this cell does ===
# 1. Generates 90 sequential calendar days starting from July 1st, 2026.
# 2. Correctly flags calendar Saturdays and Sundays as weekends, automatically adjusting their baseline consumer demand multiplier upward to 1.3.
# 3. Models realistic weather conditions for every single day, applying a randomized 25% chance of rainfall.

# === You should see ===
# 1. A table layout with columns: day_number, date, is_weekend, weather, and demand_multiplier.
# 2. A DataFrame shape of exactly (90, 5).
# 3. Clear days outnumbering rainy days in roughly a 3:1 ratio (approx. 20-25 rain days).
# 4. Weekend counts showing approximately 26 True entries (matching the calendar distribution for 90 days).
# 5. The minimum date as 2026-07-01 and maximum date as 2026-09-28.

=== DAYS DATAFRAME FIRST 10 ROWS ===


,day_number,date,is_weekend,weather,demand_multiplier
0,1,2026-07-01,False,clear,1.0
1,2,2026-07-02,False,clear,1.0
2,3,2026-07-03,False,rain,1.0
3,4,2026-07-04,True,clear,1.3
4,5,2026-07-05,True,rain,1.3
5,6,2026-07-06,False,clear,1.0
6,7,2026-07-07,False,clear,1.0
7,8,2026-07-08,False,clear,1.0
8,9,2026-07-09,False,clear,1.0
9,10,2026-07-10,False,clear,1.0



=== DATAFRAME SHAPE ===
(90, 5)

=== WEATHER COUNTS ===


,count
weather,
clear,64
rain,26



=== WEEKEND COUNTS ===


,count
is_weekend,
False,64
True,26



=== MIN AND MAX DATES ===
First simulated date: 2026-07-01
Last simulated date: 2026-09-28


In [6]:
# CELL B3: Create missions working table skeleton
# This cell builds the core structure for our delivery missions simulation, establishing
# unique mission IDs, starting timestamps, operating zones, distances, and software versions.

# Step 1: Pair every robot with every day (cross join)
# We perform a cross merge to cross-multiply all 50 robots against all 90 operating calendar days
robots_temp_df = robots_df[["robot_id"]].copy()
days_temp_df = days_df[["day_number", "date", "weather", "is_weekend", "demand_multiplier"]].copy()

# Adding a dummy key to perform a cross join in older pandas, then dropping it
robots_temp_df["dummy_key"] = 1
days_temp_df["dummy_key"] = 1

missions_work = pd.merge(robots_temp_df, days_temp_df, on="dummy_key").drop(columns=["dummy_key"])

# Step 2: Determine expected and simulated daily mission counts using a Poisson distribution
# Average target is scaled by the day's demand multiplier, and random generator draws actual counts
expected_mission_counts = AVERAGE_MISSIONS_PER_ROBOT_PER_DAY * missions_work["demand_multiplier"]
simulated_counts = random_generator.poisson(lam=expected_mission_counts)

# Step 3: Duplicate each robot-day row by the simulated mission count
# We repeat rows based on our count, reset the index, and keep the structure clean
missions_work = missions_work.loc[missions_work.index.repeat(simulated_counts)].reset_index(drop=True)

# Step 4: Draw starting hours, minutes, and seconds for each mission
# We construct hour probabilities based on whether they fall inside peak rush hours or normal hours
hours_range = np.arange(FIRST_HOUR, LAST_HOUR + 1)
hour_weights = np.where(
    np.isin(hours_range, RUSH_HOURS),
    RUSH_HOUR_DEMAND_WEIGHT,
    1.0
)
normalized_hour_weights = hour_weights / np.sum(hour_weights)

# Draw random starting hours, minutes, and seconds for all missions vectorially
total_missions = len(missions_work)
chosen_hours = random_generator.choice(
    hours_range,
    size=total_missions,
    p=normalized_hour_weights
)
chosen_minutes = random_generator.integers(0, 60, size=total_missions)
chosen_seconds = random_generator.integers(0, 60, size=total_missions)

# Construct start_time timestamps by adding time offsets to our calendar dates
start_timestamps = pd.to_datetime(missions_work["date"])
start_offsets = (
    pd.to_timedelta(chosen_hours, unit="h")
    + pd.to_timedelta(chosen_minutes, unit="m")
    + pd.to_timedelta(chosen_seconds, unit="s")
)
missions_work["start_time"] = start_timestamps + start_offsets

# Step 5: Sort by start_time and generate unique, sequential, padded mission IDs
# Sorting ensures operational continuity before we index missions from M000001
missions_work = missions_work.sort_values(by="start_time").reset_index(drop=True)
mission_indices = np.arange(1, len(missions_work) + 1)
missions_work["mission_id"] = [f"M{idx:06d}" for idx in mission_indices]

# Step 6: Assign delivery zones based on market distribution shares
# We extract the names and normalized probabilities from our ZONE_SHARES settings
zone_names = list(ZONE_SHARES.keys())
zone_probabilities = list(ZONE_SHARES.values())
missions_work["zone"] = random_generator.choice(
    zone_names,
    size=total_missions,
    p=zone_probabilities
)

# Step 7: Draw mission travel distances using a log-normal distribution
# We compute log-mean, draw distances, clip them to bounds, and round to 2 decimals
lognormal_mu = np.log(MEDIAN_DISTANCE_KM)
simulated_distances = random_generator.lognormal(
    mean=lognormal_mu,
    sigma=DISTANCE_SPREAD,
    size=total_missions
)
simulated_distances_clipped = np.clip(simulated_distances, MINIMUM_DISTANCE_KM, MAXIMUM_DISTANCE_KM)
missions_work["distance_km"] = np.round(simulated_distances_clipped, decimals=2)

# Step 8: Assign software_version dynamically based on date and robot ID numbers
# We parse numerical ID out of robot_id strings to split firmware releases
robot_numeric_ids = missions_work["robot_id"].str.extract(r"(\d+)").astype(int)[0]

# Condition A: Simulation days before first v2.2 deployment get baseline version v2.1
condition_v21 = missions_work["day_number"] < FIRST_V22_DAY

# Condition B: Days in range of v2.2 deployment up to v2.3 release get version v2.2
condition_v22 = (
    (missions_work["day_number"] >= FIRST_V22_DAY)
    & (missions_work["day_number"] < FIRST_V23_DAY)
)

# Condition C: For days starting v2.3 release, even-numbered robots get v2.3, others remain v2.2
condition_v23_even = (
    (missions_work["day_number"] >= FIRST_V23_DAY)
    & (robot_numeric_ids % 2 == 0)
)

# Default falls back to v2.2 for odd robots after the v2.3 release day
version_conditions = [condition_v21, condition_v22, condition_v23_even]
version_choices = ["v2.1", "v2.2", "v2.3"]

missions_work["software_version"] = np.select(
    version_conditions,
    version_choices,
    default="v2.2"
)

# Step 9: Establish hidden helper columns for later speed and telemetry calculations
# These variables simplify operations down the pipeline and will be dropped before final export
missions_work["hidden_start_hour"] = chosen_hours
missions_work["hidden_is_rush_hour"] = np.isin(chosen_hours, RUSH_HOURS)
missions_work["hidden_is_night"] = chosen_hours >= NIGHT_START_HOUR
missions_work["hidden_is_old_town"] = missions_work["zone"] == "Old Town"
missions_work["hidden_is_rain"] = missions_work["weather"] == "rain"

# Step 10: Calculate ideal promised_eta timestamps with speed and buffer time
# Travel duration in minutes = (distance_km / PLANNED_SPEED_KMH) * 60 + PROMISED_BUFFER_MINUTES
travel_duration_minutes = (missions_work["distance_km"] / PLANNED_SPEED_KMH) * 60.0
total_promised_duration_minutes = travel_duration_minutes + PROMISED_BUFFER_MINUTES

missions_work["promised_eta"] = (
    missions_work["start_time"]
    + pd.to_timedelta(total_promised_duration_minutes, unit="m")
)

# Step 11: Print diagnostic verification checks
print("=== SHAPE OF MISSIONS_WORK ===")
print(missions_work.shape)

print("\n=== MISSIONS_WORK FIRST 10 ROWS ===")
display(missions_work.head(10))

print("\n=== AVERAGE MISSIONS PER DAY ===")
print(f"{len(missions_work) / NUMBER_OF_DAYS:.2f}")

print("\n=== ZONE SHARE DISTRIBUTION VS SETTINGS ===")
zone_counts = missions_work["zone"].value_counts(normalize=True)
for zone_name, expected_share in ZONE_SHARES.items():
    actual_share = zone_counts.get(zone_name, 0.0)
    print(f"{zone_name:10s} -> Expected: {expected_share:.2%}, Simulated: {actual_share:.2%}")

print("\n=== SOFTWARE VERSION DISTRIBUTION ===")
display(missions_work["software_version"].value_counts(normalize=True))

print("\n=== RUSH HOUR START SHARE ===")
rush_share = missions_work["hidden_is_rush_hour"].mean()
print(f"Share of trips starting during rush hours: {rush_share:.2%}")

print("\n=== TRIP DISTANCE STATISTICS (KM) ===")
print(f"Median distance: {missions_work['distance_km'].median():.2f} km")
print(f"Mean distance:   {missions_work['distance_km'].mean():.2f} km")

# === What this cell does ===
# 1. Combines robots and days into a 4,500-row master schedule matrix, generating a dynamic stream of daily deliveries using a Poisson distribution.
# 2. Assigns timestamps weighted heavily toward commuter mealtime hours and projects destination zones matching real market shares.
# 3. Allocates flight travel distances log-normally and simulates a vectorized progressive software deployment scheme.

# === You should see ===
# 1. A dataset containing approximately 40,000 to 48,000 rows and 16 columns (including hidden helper columns).
# 2. Simulated zone shares tracking within 1% of settings (e.g. Campus and Downtown near 25.0%).
# 3. v2.1 software versions running on roughly 33.3% of records, v2.2 on 50%, and v2.3 on roughly 16.7%.
# 4. Around 65% of deliveries originating inside designated lunch and dinner rush hours.
# 5. Delivery distances showing a median value close to 1.50 km and an average/mean value close to 1.70 km.

=== SHAPE OF MISSIONS_WORK ===
(44216, 17)

=== MISSIONS_WORK FIRST 10 ROWS ===


,robot_id,day_number,date,weather,is_weekend,demand_multiplier,start_time,mission_id,zone,distance_km,software_version,hidden_start_hour,hidden_is_rush_hour,hidden_is_night,hidden_is_old_town,hidden_is_rain,promised_eta
0,R009,1,2026-07-01,clear,False,1.0,2026-07-01 08:05:14,M000001,Campus,2.04,v2.1,17,True,False,False,False,2026-07-01 08:32:29.272727270
1,R003,1,2026-07-01,clear,False,1.0,2026-07-01 08:06:34,M000002,Old Town,0.84,v2.1,19,True,False,True,False,2026-07-01 08:20:43.818181816
2,R042,1,2026-07-01,clear,False,1.0,2026-07-01 08:09:12,M000003,Riverside,4.21,v2.1,19,True,False,False,False,2026-07-01 09:00:07.636363638
3,R011,1,2026-07-01,clear,False,1.0,2026-07-01 08:15:23,M000004,Riverside,1.84,v2.1,15,False,False,False,False,2026-07-01 08:40:27.363636362
4,R049,1,2026-07-01,clear,False,1.0,2026-07-01 08:19:39,M000005,Suburb,0.88,v2.1,12,True,False,False,False,2026-07-01 08:34:15.000000000
5,R048,1,2026-07-01,clear,False,1.0,2026-07-01 08:27:37,M000006,Downtown,1.11,v2.1,19,True,False,False,False,2026-07-01 08:44:43.545454546
6,R024,1,2026-07-01,clear,False,1.0,2026-07-01 08:29:04,M000007,Campus,2.22,v2.1,17,True,False,False,False,2026-07-01 08:58:17.090909092
7,R031,1,2026-07-01,clear,False,1.0,2026-07-01 08:31:20,M000008,Campus,2.68,v2.1,18,True,False,False,False,2026-07-01 09:05:34.181818184
8,R019,1,2026-07-01,clear,False,1.0,2026-07-01 08:32:42,M000009,Riverside,0.67,v2.1,13,True,False,False,False,2026-07-01 08:45:00.545454546
9,R016,1,2026-07-01,clear,False,1.0,2026-07-01 08:34:25,M000010,Riverside,5.07,v2.1,20,False,True,False,False,2026-07-01 09:34:43.545454546



=== AVERAGE MISSIONS PER DAY ===
491.29

=== ZONE SHARE DISTRIBUTION VS SETTINGS ===
Campus     -> Expected: 25.00%, Simulated: 25.08%
Downtown   -> Expected: 25.00%, Simulated: 25.05%
Riverside  -> Expected: 20.00%, Simulated: 19.83%
Suburb     -> Expected: 18.00%, Simulated: 18.00%
Old Town   -> Expected: 12.00%, Simulated: 12.03%

=== SOFTWARE VERSION DISTRIBUTION ===


,proportion
software_version,
v2.2,0.499977
v2.1,0.331803
v2.3,0.168220



=== RUSH HOUR START SHARE ===
Share of trips starting during rush hours: 65.33%

=== TRIP DISTANCE STATISTICS (KM) ===
Median distance: 1.51 km
Mean distance:   1.70 km


In [7]:
# CELL B4: Create interventions table and link back to missions
# This cell models human-in-the-loop telemetry interventions, computes complex event probabilities
# based on rainy conditions, old cobblestone zones, and software versions, and designs our hidden analytical patterns.

# Step 1: Calculate the custom operational intervention multiplier for each mission
# We multiply baseline software penalties, rain factors, special wet-weather code glitches, and geographic zones
version_factors = missions_work["software_version"].map(VERSION_INTERVENTION_MULTIPLIER)

rain_factors = np.where(
    missions_work["hidden_is_rain"],
    RAIN_INTERVENTION_MULTIPLIER,
    1.0
)

v23_rain_factors = np.where(
    (missions_work["software_version"] == "v2.3") & missions_work["hidden_is_rain"],
    V23_IN_RAIN_EXTRA_MULTIPLIER,
    1.0
)

old_town_factors = np.where(
    missions_work["hidden_is_old_town"],
    OLD_TOWN_INTERVENTION_MULTIPLIER,
    1.0
)

night_factors = np.where(
    missions_work["hidden_is_night"],
    NIGHT_INTERVENTION_MULTIPLIER,
    1.0
)

# Compute final scalar product factor vector
hidden_intervention_multiplier = (
    version_factors
    * rain_factors
    * v23_rain_factors
    * old_town_factors
    * night_factors
)

# Step 2: Compute expected average count of intervention events
hidden_expected_interventions = (
    BASE_INTERVENTIONS_PER_KM
    * missions_work["distance_km"]
    * hidden_intervention_multiplier
)

# Step 3: Draw actual simulated intervention counts from our Poisson distribution
missions_work["hidden_intervention_count"] = random_generator.poisson(
    lam=hidden_expected_interventions
)

# Step 4: Expand missions dataset to build one row per simulated intervention event
interventions_work = missions_work.loc[
    missions_work.index.repeat(missions_work["hidden_intervention_count"])
].copy()

# Keep only the subset of columns required for downstream timing and attribution calculations
interventions_work = interventions_work[[
    "mission_id", "robot_id", "start_time",
    "distance_km", "software_version", "weather", "zone"
]].reset_index(drop=True)

# Step 5: Assign randomized timestamp offsets to interventions during active transits
# Trip durations are estimated using physical baseline speeds, and events occur randomly mid-route
travel_duration_hours = interventions_work["distance_km"] / ACTUAL_SPEED_KMH
travel_duration_minutes = travel_duration_hours * 60.0

random_trip_fractions = random_generator.random(size=len(interventions_work))
random_offsets_minutes = random_trip_fractions * travel_duration_minutes

interventions_work["intervention_time"] = (
    interventions_work["start_time"]
    + pd.to_timedelta(random_offsets_minutes, unit="m")
)

# Step 6: Vectorized assignment of diagnostic telemetry failure reasons
# We establish baseline failure category weights and dynamically scale them for rain, zones, and firmware
reason_categories = list(REASON_BASE_WEIGHTS.keys())
number_of_interventions = len(interventions_work)

# Populate an array with our base weights for each active record
weights_matrix = np.tile(
    [REASON_BASE_WEIGHTS[category] for category in reason_categories],
    (number_of_interventions, 1)
).astype(float)

# Apply selective multiplication adjustments across specific categories
# Column 0: crosswalk_confusion, Column 3: sensor_obstructed, Columns 1 & 5: blocked_path, curb_or_door
if "crosswalk_confusion" in reason_categories:
    v23_mask = (interventions_work["software_version"] == "v2.3").values
    weights_matrix[v23_mask, 0] *= V23_CROSSWALK_WEIGHT_MULTIPLIER

if "sensor_obstructed" in reason_categories:
    rain_mask = (interventions_work["weather"] == "rain").values
    weights_matrix[rain_mask, 3] *= RAIN_SENSOR_WEIGHT_MULTIPLIER

if "blocked_path" in reason_categories and "curb_or_door" in reason_categories:
    old_town_mask = (interventions_work["zone"] == "Old Town").values
    weights_matrix[old_town_mask, 1] *= OLD_TOWN_BLOCKED_AND_CURB_WEIGHT_MULTIPLIER
    weights_matrix[old_town_mask, 5] *= OLD_TOWN_BLOCKED_AND_CURB_WEIGHT_MULTIPLIER

# Trick: Normalize weights to 1.0, construct running cumulative sums, and query random thresholds
row_sums = weights_matrix.sum(axis=1, keepdims=True)
normalized_weights = weights_matrix / row_sums
cumulative_weights = np.cumsum(normalized_weights, axis=1)

random_thresholds = random_generator.random(size=(number_of_interventions, 1))
selected_indices = np.argmax(cumulative_weights > random_thresholds, axis=1)
interventions_work["reason"] = [reason_categories[idx] for idx in selected_indices]

# Step 7: Simulate intervention durations using a log-normal distribution
lognormal_sigma = INTERVENTION_DURATION_SPREAD
lognormal_mu = np.log(INTERVENTION_MEDIAN_SECONDS)
simulated_durations = random_generator.lognormal(
    mean=lognormal_mu,
    sigma=lognormal_sigma,
    size=number_of_interventions
)
simulated_durations_clipped = np.clip(
    simulated_durations,
    MIN_INTERVENTION_SECONDS,
    MAX_INTERVENTION_SECONDS
)
interventions_work["duration_sec"] = np.round(simulated_durations_clipped).astype(int)

# Step 8: Sort records by event timestamp and generate serial IDs starting from I000001
interventions_work = interventions_work.sort_values(by="intervention_time").reset_index(drop=True)
serial_indices = np.arange(1, len(interventions_work) + 1)
interventions_work["intervention_id"] = [f"I{idx:06d}" for idx in serial_indices]

# Step 9: Isolate exactly the specified final schema columns
interventions_df = interventions_work[[
    "intervention_id", "mission_id", "robot_id",
    "intervention_time", "reason", "duration_sec"
]].copy()

# Step 10: Compute total intervention seconds per mission and append back to missions_work
# We calculate totals per mission, then map them back to the original index matching missions_work
duration_totals = interventions_df.groupby("mission_id")["duration_sec"].sum()
missions_work["hidden_intervention_seconds_total"] = (
    missions_work["mission_id"].map(duration_totals).fillna(0).astype(int)
)

# Step 11: Print verification diagnostic checks
print("=== INTERVENTIONS SHAPE ===")
print(interventions_df.shape)

print("\n=== INTERVENTIONS PREVIEW (FIRST 5 ROWS) ===")
display(interventions_df.head(5))

print(f"\nTotal fleet interventions simulated: {len(interventions_df)}")

total_distance_thousands_km = missions_work["distance_km"].sum() / 1000.0
overall_rate = len(interventions_df) / total_distance_thousands_km
print(f"Overall Intervention Rate (per 1,000 km): {overall_rate:.2f}")

print("\n=== INTERVENTION RATES (PER 1,000 KM) ===")
# Helper variables to print splits easily
for version in ["v2.1", "v2.2", "v2.3"]:
    v_dist = missions_work.loc[missions_work["software_version"] == version, "distance_km"].sum() / 1000.0
    v_count = (interventions_work["software_version"] == version).sum()
    print(f"Software {version:5s} -> Rate: {v_count / v_dist:.2f}")

for w_state in ["rain", "clear"]:
    w_dist = missions_work.loc[missions_work["weather"] == w_state, "distance_km"].sum() / 1000.0
    w_count = (interventions_work["weather"] == w_state).sum()
    print(f"Weather {w_state:5s}  -> Rate: {w_count / w_dist:.2f}")

for zone_name in ZONE_SHARES.keys():
    z_dist = missions_work.loc[missions_work["zone"] == zone_name, "distance_km"].sum() / 1000.0
    z_count = (interventions_work["zone"] == zone_name).sum()
    print(f"Zone {zone_name:10s} -> Rate: {z_count / z_dist:.2f}")

zero_int_share = (missions_work["hidden_intervention_count"] == 0).mean()
print(f"\nShare of missions with zero interventions: {zero_int_share:.2%}")

print("\n=== TOP 3 INTERVENTION REASONS ===")
display(interventions_df["reason"].value_counts().head(3))

print("\n=== TOP REASON COMPARISON BY VERSION ===")
for version in ["v2.2", "v2.3"]:
    top_reason = interventions_work.loc[interventions_work["software_version"] == version, "reason"].value_counts().idxmax()
    print(f"Version {version} top reason: {top_reason}")

# === What this cell does ===
# 1. Calculates contextual intervention multipliers across 44,000+ missions, magnifying risks for rain, nighttime, or Old Town cobbles.
# 2. Replicates records based on Poisson outcomes to model individual, randomly timed, and lognormally duration-distributed human recoveries.
# 3. Vectorizes failure classifications using a normalization trick to avoid loops, and appends cumulative telemetry metadata back to our missions table.

# === You should see ===
# 1. An interventions dataset with approximately 2,000 to 3,000 total rows, structured into exactly 6 columns.
# 2. Rates showing elevated human assists on old firmware (v2.1) and rainy or old town routes (Old Town rates should approach 60+ per 1,000 km).
# 3. An overall rate of ~30-40 interventions per 1,000 km, with ~94-96% of all trips operating completely intervention-free (zero count).
# 4. v2.3 showing 'crosswalk_confusion' as its dominant issue, whereas v2.2 shows general occurrences like 'crosswalk_confusion' or 'blocked_path'.

=== INTERVENTIONS SHAPE ===
(3095, 6)

=== INTERVENTIONS PREVIEW (FIRST 5 ROWS) ===


,intervention_id,mission_id,robot_id,intervention_time,reason,duration_sec
0,I000001,M000014,R004,2026-07-01 08:58:33.072432508,low_confidence,16
1,I000002,M000010,R016,2026-07-01 09:16:01.921325404,low_confidence,45
2,I000003,M000042,R005,2026-07-01 10:38:59.083098326,crosswalk_confusion,44
3,I000004,M000054,R018,2026-07-01 11:14:50.715932422,blocked_path,20
4,I000005,M000064,R042,2026-07-01 11:18:38.530087626,crosswalk_confusion,90



Total fleet interventions simulated: 3095
Overall Intervention Rate (per 1,000 km): 41.17

=== INTERVENTION RATES (PER 1,000 KM) ===
Software v2.1  -> Rate: 38.59
Software v2.2  -> Rate: 36.54
Software v2.3  -> Rate: 60.14
Weather rain   -> Rate: 56.81
Weather clear  -> Rate: 34.38
Zone Campus     -> Rate: 37.37
Zone Downtown   -> Rate: 36.17
Zone Riverside  -> Rate: 35.37
Zone Suburb     -> Rate: 38.13
Zone Old Town   -> Rate: 73.81

Share of missions with zero interventions: 93.41%

=== TOP 3 INTERVENTION REASONS ===


,count
reason,
crosswalk_confusion,867
blocked_path,740
low_confidence,514



=== TOP REASON COMPARISON BY VERSION ===
Version v2.2 top reason: blocked_path
Version v2.3 top reason: crosswalk_confusion


In [10]:
# CELL B5: Create battery logs table
# This cell models our fleet's battery state-of-charge dynamics, calculating starting charge,
# distance-based drainage accelerated by degraded health profiles, and flagging critical system battery failures.

# Step 1: Look up each mission's robot battery health from our master robots table
# We merge robots_df to obtain 'battery_health' and store it in missions_work as a hidden helper column
missions_work = missions_work.merge(
    robots_df[["robot_id", "battery_health"]],
    on="robot_id",
    how="left"
).rename(columns={"battery_health": "hidden_battery_health"})

# Step 2: Simulate starting battery charge levels dynamically
# Note: This starting charge is drawn as a random float between START_CHARGE_RANGE bounds for each mission.
# This is an operational simplification, because real-world robots would recharge or hot-swap battery modules between trips.
charge_start_min = START_CHARGE_RANGE[0]
charge_start_max = START_CHARGE_RANGE[1]
random_charge_starts = random_generator.uniform(
    low=charge_start_min,
    high=charge_start_max,
    size=len(missions_work)
)
missions_work["charge_start_pct"] = np.round(random_charge_starts, decimals=1)

# Step 3: Calculate the battery percentage consumed during each transit
# Standard drainage rate is scaled by the robot's health coefficient. A tired battery (e.g. 0.65 health) drains faster.
hidden_charge_used_pct = (
    missions_work["distance_km"]
    * BATTERY_DRAIN_PERCENT_PER_KM
) / missions_work["hidden_battery_health"]

# Step 4: Calculate the remaining battery charge at the end of the trip
# The final value is clipped to ensure it never drops below 0.0% charge capacity
charge_end_raw = missions_work["charge_start_pct"] - hidden_charge_used_pct
missions_work["charge_end_pct"] = np.round(
    np.maximum(charge_end_raw, 0.0),
    decimals=1
)

# Step 5: Identify critical failures due to battery exhaustion
# A critical failure is flagged if the remaining capacity falls below our safety buffer limit (5%)
missions_work["hidden_battery_failed"] = (
    missions_work["charge_end_pct"] < BATTERY_FAILURE_THRESHOLD_PCT
)

# Step 6: Isolate and extract exactly the specified final schema columns
battery_logs_df = missions_work[[
    "mission_id",
    "robot_id",
    "charge_start_pct",
    "charge_end_pct"
]].copy()

# Step 7: Run verification diagnostic checks and comparative fleet audits
print("=== BATTERY LOGS SHAPE ===")
print(battery_logs_df.shape)

print("=== BATTERY LOGS PREVIEW (FIRST 5 ROWS) ===")
display(battery_logs_df.head(5))

# Calculate operational stats for tired robots vs. healthy counterparts
is_tired_mask = missions_work["robot_id"].isin(TIRED_BATTERY_ROBOT_IDS)

# A: Average battery charge used per kilometer
# Charge Used = Start Pct - End Pct
missions_work["tmp_charge_used"] = missions_work["charge_start_pct"] - missions_work["charge_end_pct"]
tired_used_per_km = (
    missions_work.loc[is_tired_mask, "tmp_charge_used"].sum()
    / missions_work.loc[is_tired_mask, "distance_km"].sum()
)
normal_used_per_km = (
    missions_work.loc[~is_tired_mask, "tmp_charge_used"].sum()
    / missions_work.loc[~is_tired_mask, "distance_km"].sum()
)

print("=== BATTERY PERFORMANCE METRICS ===")
print(f"Average charge drain per km (Tired Robots):  {tired_used_per_km:.3f}%")
print(f"Average charge drain per km (Other Robots):  {normal_used_per_km:.3f}%")

# B: Failure occurrences and relative rate statistics
total_battery_failures = missions_work["hidden_battery_failed"].sum()
tired_failures = missions_work.loc[is_tired_mask, "hidden_battery_failed"].sum()
normal_failures = missions_work.loc[~is_tired_mask, "hidden_battery_failed"].sum()

tired_rate = missions_work.loc[is_tired_mask, "hidden_battery_failed"].mean()
normal_rate = missions_work.loc[~is_tired_mask, "hidden_battery_failed"].mean()
share_of_failures_from_tired = (tired_failures / total_battery_failures) if total_battery_failures > 0 else 0.0

print(f"\nTotal fleet battery failures detected: {total_battery_failures}")
print(f"Battery failure rate per trip (Tired Robots):  {tired_rate:.2%}")
print(f"Battery failure rate per trip (Other Robots):  {normal_rate:.2%}")
print(f"Share of all battery failures from tired group: {share_of_failures_from_tired:.2%}")

# Drop our temporary metrics calculation helper
missions_work = missions_work.drop(columns=["tmp_charge_used"])

# === What this cell does ===
# 1. Joins robot operational health scores to missions to scale consumption rates.
# 2. Generates randomized start charges and models depletion proportional to travel distances.
# 3. Flags low-power failures and exports a clean dataset while auditing performance across hardware groups.

# === You should see ===
# 1. A table shape of exactly (44216, 4) matches the total mission log size.
# 2. Average charge drain showing approximately 7.6% per km for degraded hardware versus ~5.4% for healthy robots.
# 3. Extremely high relative failure rates among our tired robot sample due to faster depletion.

=== BATTERY LOGS SHAPE ===
(44216, 4)
=== BATTERY LOGS PREVIEW (FIRST 5 ROWS) ===


,mission_id,robot_id,charge_start_pct,charge_end_pct
0,M000001,R009,32.1,20.5
1,M000002,R003,55.3,50.9
2,M000003,R042,66.7,44.8
3,M000004,R011,73.4,62.7
4,M000005,R049,85.7,80.9


=== BATTERY PERFORMANCE METRICS ===
Average charge drain per km (Tired Robots):  7.677%
Average charge drain per km (Other Robots):  5.432%

Total fleet battery failures detected: 118
Battery failure rate per trip (Tired Robots):  1.05%
Battery failure rate per trip (Other Robots):  0.18%
Share of all battery failures from tired group: 38.98%


In [11]:
# CELL B6: Arrival times and mission status
# This cell models physical robot driving speeds, applies contextual environmental transit slowdowns,
# determines arrival timestamps, and resolves final mission completion/failure status with detailed audits.

# Step 1: Draw a randomized speed variation factor for each mission
# This represents small physical variations in robot motors or pavement friction conditions
speed_factors = random_generator.uniform(
    low=SPEED_RANDOM_RANGE[0],
    high=SPEED_RANDOM_RANGE[1],
    size=len(missions_work)
)
missions_work["speed_factor"] = speed_factors
missions_work["actual_speed_kmh"] = ACTUAL_SPEED_KMH * missions_work["speed_factor"]

# Step 2: Compute the multiplicative transit slowdown factor
# The total slowdown is the product of weather, spatial zone, and peak traffic time multipliers
rain_slowdowns = np.where(missions_work["weather"] == "rain", RAIN_SLOWDOWN, 1.0)
old_town_slowdowns = np.where(missions_work["zone"] == "Old Town", OLD_TOWN_SLOWDOWN, 1.0)
rush_hour_slowdowns = np.where(missions_work["hidden_is_rush_hour"], RUSH_HOUR_SLOWDOWN, 1.0)

missions_work["hidden_slowdown"] = rain_slowdowns * old_town_slowdowns * rush_hour_slowdowns

# Step 3: Compute total physical transit travel duration in minutes
# Time = (distance / actual_speed) * 60 minutes * combined environmental slowdown + cumulative remote operator delays
missions_work["hidden_travel_minutes"] = (
    (missions_work["distance_km"] / missions_work["actual_speed_kmh"]) * 60.0 * missions_work["hidden_slowdown"]
) + (missions_work["hidden_intervention_seconds_total"] / 60.0)

# Step 4: Calculate the actual arrival timestamp
# We add the calculated travel duration minutes to the mission's initial start_time
missions_work["actual_arrival"] = (
    missions_work["start_time"]
    + pd.to_timedelta(missions_work["hidden_travel_minutes"], unit="m")
)

# Step 5: Assign final trip status and failure reasons using vectorized selection
# Draw random arrays to determine generic hardware failures and assign their specific codes
random_failures_draws = random_generator.random(size=len(missions_work))
random_failure_reason_selections = random_generator.choice(
    RANDOM_FAILURE_REASONS,
    size=len(missions_work)
)

# Condition A: Mission fails if the battery depleted below the critical safety threshold
condition_battery_failed = missions_work["hidden_battery_failed"].values

# Condition B: Mission fails due to random hardware faults or getting physically stuck
condition_random_failed = (~missions_work["hidden_battery_failed"].values) & (random_failures_draws < RANDOM_FAILURE_RATE)

# Setup condition arrays and matching value maps for np.select
status_conditions = [
    condition_battery_failed,
    condition_random_failed
]

status_choices = ["failed", "failed"]
reason_choices = ["battery_low", random_failure_reason_selections]

# Populate final status and failure columns, defaulting to completed with no failure reason
missions_work["status"] = np.select(status_conditions, status_choices, default="completed")
missions_work["failure_reason"] = np.select(status_conditions, reason_choices, default=None)

# Step 6: Nullify actual arrival timestamps for trips that did not complete successfully
# Failed missions do not arrive, so we overwrite their timestamps with pandas Not-a-Time (NaT)
missions_work.loc[
    missions_work["status"] == "failed",
    "actual_arrival"
] = pd.NaT

# Step 7: Print diagnostic audits and on-time performance splits
print("=== MISSION STATUS DISTRIBUTION ===")
status_distribution = missions_work["status"].value_counts(normalize=True)
print(status_distribution)

print("\n=== FAILURE REASON COUNTS ===")
failure_reason_counts = missions_work["failure_reason"].value_counts(dropna=False)
print(failure_reason_counts)

# On-time calculations apply only to completed missions. On-time is defined as actual_arrival <= promised_eta
is_completed = missions_work["status"] == "completed"
missions_work["is_on_time"] = np.where(
    is_completed,
    missions_work["actual_arrival"] <= missions_work["promised_eta"],
    False
)

overall_otp = missions_work.loc[is_completed, "is_on_time"].mean()
print(f"\nOverall On-Time Performance (Completed Only): {overall_otp:.2%}")

print("\n=== ON-TIME PERFORMANCE BY SPLITS ===")
# On-time performance by weather
for w_state in ["clear", "rain"]:
    subset_mask = is_completed & (missions_work["weather"] == w_state)
    split_otp = missions_work.loc[subset_mask, "is_on_time"].mean()
    print(f"Weather {w_state:5s}  -> OTP: {split_otp:.2%}")

# On-time performance by zone
for zone_name in ZONE_SHARES.keys():
    subset_mask = is_completed & (missions_work["zone"] == zone_name)
    split_otp = missions_work.loc[subset_mask, "is_on_time"].mean()
    print(f"Zone {zone_name:10s} -> OTP: {split_otp:.2%}")

# On-time performance by rush hour
for is_rush in [True, False]:
    subset_mask = is_completed & (missions_work["hidden_is_rush_hour"] == is_rush)
    split_otp = missions_work.loc[subset_mask, "is_on_time"].mean()
    rush_label = "Rush Hour" if is_rush else "Normal Hour"
    print(f"{rush_label:11s} -> OTP: {split_otp:.2%}")

# On-time performance for the BASELINE group (clear weather, not Old Town, not rush hour)
is_baseline = (
    (missions_work["weather"] == "clear")
    & (missions_work["zone"] != "Old Town")
    & (~missions_work["hidden_is_rush_hour"])
)
baseline_otp = missions_work.loc[is_completed & is_baseline, "is_on_time"].mean()
print(f"\nBaseline Group On-Time Performance: {baseline_otp:.2%}")

# === What this cell does ===
# 1. Models actual travel speed deviations and compounds delay multipliers (rain slowdown, old cobblestone zones, rush-hour).
# 2. Computes the exact timestamp the robot would reach the customer or marks them failed if halted by low batteries or generic hardware faults.
# 3. Measures operational Service Level Agreements (SLAs) overall and across multiple strategic and environmental groupings.

# === You should see ===
# 1. Complete status rate near ~97.1%, with battery failures accounting for ~118 trips and random hardware errors for ~660+ trips.
# 2. Overall On-Time Performance (OTP) close to ~83-85%.
# 3. Baseline group OTP (with no traffic or weather slowdowns) approaching an impressive ~98-99%.
# 4. Severe performance degradation on rainy or congested routes, dropping OTP below 50-60%.

=== MISSION STATUS DISTRIBUTION ===
status
completed    0.982224
failed       0.017776
Name: proportion, dtype: float64

=== FAILURE REASON COUNTS ===
failure_reason
None                43430
stuck_or_blocked      343
hardware_fault        325
battery_low           118
Name: count, dtype: int64

Overall On-Time Performance (Completed Only): 51.82%

=== ON-TIME PERFORMANCE BY SPLITS ===
Weather clear  -> OTP: 64.14%
Weather rain   -> OTP: 23.14%
Zone Campus     -> OTP: 58.70%
Zone Downtown   -> OTP: 57.59%
Zone Riverside  -> OTP: 58.02%
Zone Suburb     -> OTP: 58.37%
Zone Old Town   -> OTP: 5.57%
Rush Hour   -> OTP: 41.64%
Normal Hour -> OTP: 70.99%

Baseline Group On-Time Performance: 93.86%


In [14]:
# CELL B7: Final tables and planted dirty data
# This cell shapes our final delivery missions schema, exports an untouched golden copy,
# and plants realistic telemetry data quality issues (duplicates, zero distances, negative durations).

# Step 1: Build the final missions dataframe with exact column alignment and rounded timestamps
# Timestamps are rounded to whole seconds, leaving behind all 'hidden_' helper attributes
missions_df = missions_work[[
    "mission_id",
    "robot_id",
    "zone",
    "start_time",
    "distance_km",
    "promised_eta",
    "actual_arrival",
    "status",
    "failure_reason",
    "software_version",
    "weather"
]].copy()

missions_df["start_time"] = missions_df["start_time"].dt.round("s")
missions_df["promised_eta"] = missions_df["promised_eta"].dt.round("s")
missions_df["actual_arrival"] = missions_df["actual_arrival"].dt.round("s")

# Step 2: Make an untouched golden copy to serve as our evaluation answer key
# This copy remains strictly memory-bound and is never saved to external storage
missions_before_dirt_df = missions_df.copy()

# Step 3: Draw different row indices to receive structured dirty data corruptions
# We ensure no overlap by sampling unique locations across the entire index length
total_rows = len(missions_df)
total_needed_dirty_records = NUMBER_OF_DUPLICATE_MISSIONS + NUMBER_OF_NEGATIVE_DURATIONS + NUMBER_OF_ZERO_DISTANCES

all_sampled_indices = random_generator.choice(
    np.arange(total_rows),
    size=total_needed_dirty_records,
    replace=False
)

# Split our single sample array into three isolated groups
group_neg_duration_indices = list(all_sampled_indices[:NUMBER_OF_NEGATIVE_DURATIONS])
group_zero_distance_indices = list(all_sampled_indices[NUMBER_OF_NEGATIVE_DURATIONS : NUMBER_OF_NEGATIVE_DURATIONS + NUMBER_OF_ZERO_DISTANCES])
group_duplicate_indices = list(all_sampled_indices[NUMBER_OF_NEGATIVE_DURATIONS + NUMBER_OF_ZERO_DISTANCES:])

# Rule: If a chosen row for negative durations is not a 'completed' mission, replace it with a completed one
completed_only_indices = np.where(missions_df["status"] == "completed")[0]
for i, idx in enumerate(group_neg_duration_indices):
    if missions_df.loc[idx, "status"] != "completed":
        # Sample a fresh random index from our completed array that is not already chosen in any group
        available_indices = np.setdiff1d(completed_only_indices, all_sampled_indices)
        new_idx = random_generator.choice(available_indices)
        # Swap the index out and track it in our sampling array
        group_neg_duration_indices[i] = new_idx
        all_sampled_indices = np.append(all_sampled_indices, new_idx)

# Step 4: Inject negative durations (arrival occurs before departure)
# For these completed rows, set actual_arrival to start_time minus a random 1 to 30 minutes
random_subtraction_minutes = random_generator.integers(1, 31, size=len(group_neg_duration_indices))
misions_neg_start_times = missions_df.loc[group_neg_duration_indices, "start_time"]
missions_df.loc[group_neg_duration_indices, "actual_arrival"] = (
    misions_neg_start_times - pd.to_timedelta(random_subtraction_minutes, unit="m")
)

# Step 5: Inject zeroed distances
# For these rows, set the total distance_km field to 0.0
missions_df.loc[group_zero_distance_indices, "distance_km"] = 0.0

# Step 6: Inject exact row duplicates
# Copy selected rows and append them directly to the bottom of the table (preserving primary key duplicates)
duplicated_rows_df = missions_df.iloc[group_duplicate_indices].copy()
missions_df = pd.concat([missions_df, duplicated_rows_df], ignore_index=True)

# Step 7: Log planted error IDs into a master dictionary
# This serves as a secure record for testing cleansing pipelines later
planted_errors_log = {
    "negative_durations_mission_ids": list(missions_before_dirt_df.iloc[group_neg_duration_indices]["mission_id"]),
    "zero_distances_mission_ids": list(missions_before_dirt_df.iloc[group_zero_distance_indices]["mission_id"]),
    "duplicates_mission_ids": list(missions_before_dirt_df.iloc[group_duplicate_indices]["mission_id"])
}

# Step 8: Print checks to verify the structured anomalies and table profiles
print("=== TARGET TABLE SHAPES ===")
print(f"Missions (With Dirty Data): {missions_df.shape}")
print(f"Robots Master Table:        {robots_df.shape}")
print(f"Interventions Table:        {interventions_df.shape}")
print(f"Battery Logs Table:         {battery_logs_df.shape}")

print("\n=== DATA QUALITY METRIC CHECKS ===")
# A: Exact duplicates count (checks complete rows matching on all columns)
exact_dup_count = missions_df.duplicated(keep=False).sum()
print(f"Count of exact row duplicates (including original and copy): {exact_dup_count}")

# B: Negative durations count
negative_duration_mask = missions_df["actual_arrival"] < missions_df["start_time"]
negative_duration_count = negative_duration_mask.sum()
print(f"Count of trips arriving earlier than start time:              {negative_duration_count}")

# C: Zero distance count
zero_distance_count = (missions_df["distance_km"] == 0.0).sum()
print(f"Count of trips with 0.0 km recorded distance:                 {zero_distance_count}")

# === What this cell does ===
# 1. Filters, rounds, and formats missions_df, creating an untouched reference answer key in memory.
# 2. Selects isolated groups of indices to corrupt with negative trip runtimes and null distances.
# 3. Appends twenty exact duplicates of existing rows to test primary key deduplication routines, auditing final tables.

# === You should see ===
# 1. Missions table size equal to 44,236 (original base) + 20 (duplicates) = 44,256 rows total.
# 2. Exact duplicate count showing exactly 40 records (20 original records + their 20 cloned duplicates).
# 3. Negative arrival checks showing exactly 30 records, and zero-distance checks showing exactly 25 records.

=== TARGET TABLE SHAPES ===
Missions (With Dirty Data): (44236, 11)
Robots Master Table:        (50, 3)
Interventions Table:        (3095, 6)
Battery Logs Table:         (44216, 4)

=== DATA QUALITY METRIC CHECKS ===
Count of exact row duplicates (including original and copy): 40
Count of trips arriving earlier than start time:              30
Count of trips with 0.0 km recorded distance:                 25


In [15]:
# CELL B8: Pattern check against the answer key
# This cell executes programmatic pattern checks using clean data frames (missions_before_dirt_df, interventions_df, battery_logs_df)
# to verify the embedded experimental design and outputs an executive Answer Key verification matrix.

# Helper calculation: Map intervention counts and total operational hours back to missions_before_dirt_df
# Note: This operates on the clean uncorrupted golden dataframe 'missions_before_dirt_df'
m_clean = missions_before_dirt_df.copy()

# Group clean interventions and map counts and durations to the baseline missions
clean_int_counts = interventions_df.groupby("mission_id").size()
clean_int_secs = interventions_df.groupby("mission_id")["duration_sec"].sum()

m_clean["intervention_count"] = m_clean["mission_id"].map(clean_int_counts).fillna(0).astype(int)
m_clean["intervention_seconds"] = m_clean["mission_id"].map(clean_int_secs).fillna(0).astype(int)

# Group battery log changes to find initial and final charge differentials
m_clean = m_clean.merge(
    battery_logs_df[["mission_id", "charge_start_pct", "charge_end_pct"]],
    on="mission_id",
    how="left"
)
m_clean["charge_used_pct"] = m_clean["charge_start_pct"] - m_clean["charge_end_pct"]

# Helper flag for night missions based on starting hour
m_clean["start_hour"] = pd.to_datetime(m_clean["start_time"]).dt.hour
m_clean["is_night"] = m_clean["start_hour"] >= NIGHT_START_HOUR

# FIX: Map 'is_weekend' back using days_df based on calendar dates
days_mapping = days_df.set_index("date")["is_weekend"]
m_clean["is_weekend"] = m_clean["start_time"].dt.date.map(days_mapping)

# -----------------------------------------------------
# Step 1: Interventions per 1,000 km by software_version
# -----------------------------------------------------
print("=== 1. INTERVENTIONS PER 1,000 KM BY SOFTWARE VERSION ===")
rate_by_version = {}
for ver in ["v2.1", "v2.2", "v2.3"]:
    subset = m_clean[m_clean["software_version"] == ver]
    total_km = subset["distance_km"].sum()
    total_ints = subset["intervention_count"].sum()
    rate = (total_ints / total_km) * 1000.0
    rate_by_version[ver] = rate
    print(f"Software {ver:5s} -> Rate: {rate:.2f} per 1,000 km")

# -----------------------------------------------------
# Step 2: Environmental splits (Rain, Zone, Night)
# -----------------------------------------------------
print("\n=== 2. ENVIRONMENTAL SPLITS (PER 1,000 KM) ===")
# Rain vs Clear
rain_subset = m_clean[m_clean["weather"] == "rain"]
clear_subset = m_clean[m_clean["weather"] == "clear"]
rain_rate = (rain_subset["intervention_count"].sum() / rain_subset["distance_km"].sum()) * 1000.0
clear_rate = (clear_subset["intervention_count"].sum() / clear_subset["distance_km"].sum()) * 1000.0
print(f"Weather Rain:   {rain_rate:.2f} | Clear: {clear_rate:.2f} (Ratio: {rain_rate/clear_rate:.2f})")

# Zones
rate_by_zone = {}
for z in ZONE_SHARES.keys():
    subset = m_clean[m_clean["zone"] == z]
    rate = (subset["intervention_count"].sum() / subset["distance_km"].sum()) * 1000.0
    rate_by_zone[z] = rate
    print(f"Zone {z:10s} -> Rate: {rate:.2f} per 1,000 km")

# Night vs Day
night_subset = m_clean[m_clean["is_night"]]
day_subset = m_clean[~m_clean["is_night"]]
night_rate = (night_subset["intervention_count"].sum() / night_subset["distance_km"].sum()) * 1000.0
day_rate = (day_subset["intervention_count"].sum() / day_subset["distance_km"].sum()) * 1000.0
print(f"Night-time:     {night_rate:.2f} | Day-time: {day_rate:.2f} (Ratio: {night_rate/day_rate:.2f})")

# -----------------------------------------------------
# Step 3: Last-30-Days Software Version A/B Test (Day 61+)
# -----------------------------------------------------
print("\n=== 3. LAST 30 DAYS SOFTWARE A/B TEST (DAY 61+) ===")
# Retrieve day numbers by calculating calendar date offsets
base_date_parsed = pd.to_datetime(START_DATE)
m_clean["day_number"] = (pd.to_datetime(m_clean["start_time"]) - base_date_parsed).dt.days + 1

ab_subset = m_clean[m_clean["day_number"] >= FIRST_V23_DAY]
rate_v22_ab = (ab_subset[ab_subset["software_version"] == "v2.2"]["intervention_count"].sum() / ab_subset[ab_subset["software_version"] == "v2.2"]["distance_km"].sum()) * 1000.0
rate_v23_ab = (ab_subset[ab_subset["software_version"] == "v2.3"]["intervention_count"].sum() / ab_subset[ab_subset["software_version"] == "v2.3"]["distance_km"].sum()) * 1000.0
print(f"Version v2.2 -> Rate: {rate_v22_ab:.2f} per 1,000 km")
print(f"Version v2.3 -> Rate: {rate_v23_ab:.2f} per 1,000 km (Ratio: {rate_v23_ab/rate_v22_ab:.2f})")

# -----------------------------------------------------
# Step 4: Top 3 reasons for each software version
# -----------------------------------------------------
print("\n=== 4. TOP 3 FAILURE REASONS BY SOFTWARE VERSION ===")
# Merge clean interventions back to match version information
int_with_meta = interventions_df.merge(m_clean[["mission_id", "software_version"]], on="mission_id", how="left")
for ver in ["v2.1", "v2.2", "v2.3"]:
    top_reasons = int_with_meta[int_with_meta["software_version"] == ver]["reason"].value_counts().head(3)
    print(f"\nVersion {ver} Top Reasons:")
    for r_name, r_cnt in top_reasons.items():
        print(f" - {r_name:20s}: {r_cnt}")

# -----------------------------------------------------
# Step 5: Autonomy Rate Overall and by software_version
# -----------------------------------------------------
print("\n=== 5. AUTONOMY RATES ===")
# Autonomy Rate is calculated as: 1.0 - (Total human-operator intervention time / Total trip time)
# Standard duration is distance divided by actual physical speed (ACTUAL_SPEED_KMH)
m_clean["ideal_travel_hours"] = m_clean["distance_km"] / ACTUAL_SPEED_KMH
m_clean["ideal_travel_seconds"] = m_clean["ideal_travel_hours"] * 3600.0

total_travel_secs = m_clean["ideal_travel_seconds"].sum()
total_int_secs = m_clean["intervention_seconds"].sum()
overall_autonomy = 1.0 - (total_int_secs / total_travel_secs)
print(f"Overall Fleet Autonomy: {overall_autonomy:.4%}")

for ver in ["v2.1", "v2.2", "v2.3"]:
    sub = m_clean[m_clean["software_version"] == ver]
    ver_autonomy = 1.0 - (sub["intervention_seconds"].sum() / sub["ideal_travel_seconds"].sum())
    print(f"Software {ver:5s} Autonomy: {ver_autonomy:.4%}")

# -----------------------------------------------------
# Step 6: On-time % (Completed missions only)
# -----------------------------------------------------
print("\n=== 6. ON-TIME PERFORMANCE (COMPLETED ONLY) ===")
m_clean_completed = m_clean[m_clean["status"] == "completed"].copy()
m_clean_completed["is_on_time"] = m_clean_completed["actual_arrival"] <= m_clean_completed["promised_eta"]

# By Weather
rain_otp = m_clean_completed[m_clean_completed["weather"] == "rain"]["is_on_time"].mean()
clear_otp = m_clean_completed[m_clean_completed["weather"] == "clear"]["is_on_time"].mean()
print(f"Weather Clear OTP: {clear_otp:.2%} | Rain OTP: {rain_otp:.2%} (Diff: {clear_otp - rain_otp:.2%})")

# By Zone
zone_otp_values = {}
for z in ZONE_SHARES.keys():
    z_otp = m_clean_completed[m_clean_completed["zone"] == z]["is_on_time"].mean()
    zone_otp_values[z] = z_otp
    print(f"Zone {z:10s} OTP: {z_otp:.2%}")

# By Start Hour (Rush vs Non-Rush)
# Note: This checks whether the hour is explicitly contained within the RUSH_HOURS list
is_rush_otp = m_clean_completed[m_clean_completed["start_hour"].isin(RUSH_HOURS)]["is_on_time"].mean()
not_rush_otp = m_clean_completed[~m_clean_completed["start_hour"].isin(RUSH_HOURS)]["is_on_time"].mean()
print(f"Rush Hour OTP:     {is_rush_otp:.2%} | Normal Hour OTP: {not_rush_otp:.2%}")

# -----------------------------------------------------
# Step 7: Average battery charge used per km by robot
# -----------------------------------------------------
print("\n=== 7. TOP 8 ROBOTS BY BATTERY DEPLETION RATE (PER KM) ===")
robot_drain = m_clean.groupby("robot_id").agg(
    total_charge_used=("charge_used_pct", "sum"),
    total_km=("distance_km", "sum")
)
robot_drain["drain_per_km"] = robot_drain["total_charge_used"] / robot_drain["total_km"]
top_8_drain = robot_drain.sort_values(by="drain_per_km", ascending=False).head(8)
display(top_8_drain)

# -----------------------------------------------------
# Step 8: Average missions per day on weekdays vs. weekends
# -----------------------------------------------------
print("\n=== 8. DAILY MISSION DEMAND BY CALENDAR TYPE ===")
weekend_mask = m_clean["is_weekend"]

# Retrieve exact unique calendar date counts across classes to divide correctly
m_clean["date_only"] = m_clean["start_time"].dt.date
total_weekend_days = m_clean.loc[weekend_mask, "date_only"].nunique()
total_weekday_days = m_clean.loc[~weekend_mask, "date_only"].nunique()

weekend_missions_avg = len(m_clean[weekend_mask]) / total_weekend_days
weekday_missions_avg = len(m_clean[~weekend_mask]) / total_weekday_days
print(f"Weekday Average: {weekday_missions_avg:.2f} missions/day")
print(f"Weekend Average: {weekend_missions_avg:.2f} missions/day (Ratio: {weekend_missions_avg/weekday_missions_avg:.2f})")

# -----------------------------------------------------
# Step 9: Programmatic Hypothesis Verification Matrix
# -----------------------------------------------------
print("\n=== SYSTEM HYPOTHESIS ANSWER KEY MATRIX ===")

# Helper lists to build our validation dataframe
patterns = []
measured = []
expected = []
status = []

# Pattern 1: Version-based regression
cond_1 = (rate_by_version["v2.2"] < rate_by_version["v2.1"]) and (rate_by_version["v2.3"] > rate_by_version["v2.1"])
patterns.append("Version Interventions Check")
measured.append(f"v2.1={rate_by_version['v2.1']:.1f}, v2.2={rate_by_version['v2.2']:.1f}, v2.3={rate_by_version['v2.3']:.1f}")
expected.append("v2.2 < v2.1 < v2.3")
status.append("PASS" if cond_1 else "FAIL")

# Pattern 2: Rain impact
rain_clear_ratio = rain_rate / clear_rate
cond_2 = 1.3 <= rain_clear_ratio <= 1.9
patterns.append("Rain/Clear Risk Ratio")
measured.append(f"{rain_clear_ratio:.2f}")
expected.append("Ratio between 1.3 and 1.9")
status.append("PASS" if cond_2 else "FAIL")

# Pattern 3: Old town penalty
other_zones_avg = np.mean([rate_by_zone[z] for z in rate_by_zone if z != "Old Town"])
old_town_ratio = rate_by_zone["Old Town"] / other_zones_avg
cond_3 = 1.6 <= old_town_ratio <= 2.5
patterns.append("Old Town / Other Zones Ratio")
measured.append(f"{old_town_ratio:.2f}")
expected.append("Ratio between 1.6 and 2.5")
status.append("PASS" if cond_3 else "FAIL")

# Pattern 4: Night visibility penalty
night_ratio = night_rate / day_rate
cond_4 = 1.1 <= night_ratio <= 1.5
patterns.append("Night / Day Risk Ratio")
measured.append(f"{night_ratio:.2f}")
expected.append("Ratio between 1.1 and 1.5")
status.append("PASS" if cond_4 else "FAIL")

# Pattern 5: v2.3 Rain Regression (Day 61+)
ab_test_ratio = rate_v23_ab / rate_v22_ab
cond_5 = ab_test_ratio > 1.3
patterns.append("Last-30-Days v2.3 vs v2.2 Ratio")
measured.append(f"{ab_test_ratio:.2f}")
expected.append("Ratio above 1.3")
status.append("PASS" if cond_5 else "FAIL")

# Pattern 6: Top reason under v2.3 software
v23_top_reason = int_with_meta[int_with_meta["software_version"] == "v2.3"]["reason"].value_counts().idxmax()
cond_6 = v23_top_reason == "crosswalk_confusion"
patterns.append("Version v2.3 Dominant Issue")
measured.append(v23_top_reason)
expected.append("crosswalk_confusion")
status.append("PASS" if cond_6 else "FAIL")

# Pattern 7: Rain SLA penalty
rain_sla_diff = (clear_otp - rain_otp) * 100.0
cond_7 = rain_sla_diff >= 5.0
patterns.append("Rain SLA Penalty Points")
measured.append(f"{rain_sla_diff:.2f} pts")
expected.append("At least 5 points reduction")
status.append("PASS" if cond_7 else "FAIL")

# Pattern 8: Old Town lowest SLA
lowest_zone = min(zone_otp_values, key=zone_otp_values.get)
best_zone = max(zone_otp_values, key=zone_otp_values.get)
zone_diff = (zone_otp_values[best_zone] - zone_otp_values["Old Town"]) * 100.0
cond_8 = (lowest_zone == "Old Town") and (zone_diff >= 10.0)
patterns.append("Old Town SLA Gap")
measured.append(f"Lowest={lowest_zone}, Gap={zone_diff:.2f} pts")
expected.append("Old Town is lowest, at least 10 pts below best")
status.append("PASS" if cond_8 else "FAIL")

# Pattern 9: Rush hour SLA impact
cond_9 = is_rush_otp < not_rush_otp
patterns.append("Rush Hour SLA Reduction")
measured.append(f"Rush={is_rush_otp:.2%}, Normal={not_rush_otp:.2%}")
expected.append("Rush Hour OTP < Normal Hour OTP")
status.append("PASS" if cond_9 else "FAIL")

# Pattern 10: Tired battery hardware profile
top_5_drain_robots = list(top_8_drain.head(5).index)
cond_10 = set(top_5_drain_robots) == set(TIRED_BATTERY_ROBOT_IDS)
patterns.append("Tired Battery Robot Drain Signature")
measured.append(str(top_5_drain_robots))
expected.append(str(TIRED_BATTERY_ROBOT_IDS))
status.append("PASS" if cond_10 else "FAIL")

# Pattern 11: Weekend user demand volume expansion
weekend_demand_ratio = weekend_missions_avg / weekday_missions_avg
cond_11 = weekend_demand_ratio >= 1.2
patterns.append("Weekend Volumetric Expansion Ratio")
measured.append(f"{weekend_demand_ratio:.2f}")
expected.append("Ratio at least 1.2")
status.append("PASS" if cond_11 else "FAIL")

# Output compiled validation table to console
answer_key_matrix = pd.DataFrame({
    "pattern": patterns,
    "measured value": measured,
    "expected rule": expected,
    "status": status
})

display(answer_key_matrix)

# === What this cell does ===
# 1. Calculates multiple diagnostic metrics on the clean datasets including software, weather, night, and zone subsets.
# 2. Tracks physical battery depletion parameters per kilometer for each of our 50 autonomous robots.
# 3. Validates 11 core hypotheses of the simulation, rendering a structured validation matrix labeled PASS/FAIL.

=== 1. INTERVENTIONS PER 1,000 KM BY SOFTWARE VERSION ===
Software v2.1  -> Rate: 38.59 per 1,000 km
Software v2.2  -> Rate: 36.54 per 1,000 km
Software v2.3  -> Rate: 60.14 per 1,000 km

=== 2. ENVIRONMENTAL SPLITS (PER 1,000 KM) ===
Weather Rain:   56.81 | Clear: 34.38 (Ratio: 1.65)
Zone Campus     -> Rate: 37.37 per 1,000 km
Zone Downtown   -> Rate: 36.17 per 1,000 km
Zone Riverside  -> Rate: 35.37 per 1,000 km
Zone Suburb     -> Rate: 38.13 per 1,000 km
Zone Old Town   -> Rate: 73.81 per 1,000 km
Night-time:     44.09 | Day-time: 40.88 (Ratio: 1.08)

=== 3. LAST 30 DAYS SOFTWARE A/B TEST (DAY 61+) ===
Version v2.2 -> Rate: 34.23 per 1,000 km
Version v2.3 -> Rate: 60.14 per 1,000 km (Ratio: 1.76)

=== 4. TOP 3 FAILURE REASONS BY SOFTWARE VERSION ===

Version v2.1 Top Reasons:
 - blocked_path        : 242
 - crosswalk_confusion : 220
 - low_confidence      : 195

Version v2.2 Top Reasons:
 - blocked_path        : 377
 - crosswalk_confusion : 271
 - low_confidence      : 234

Version 

,total_charge_used,total_km,drain_per_km
robot_id,,,
R023,11874.2,1544.64,7.687358
R031,11940.2,1553.65,7.685257
R007,11122.2,1447.76,7.682351
R044,11705.4,1526.96,7.665820
R019,11166.9,1457.50,7.661681
R010,8920.3,1517.76,5.877280
R034,8646.8,1486.94,5.815164
R011,8580.9,1475.79,5.814445



=== 8. DAILY MISSION DEMAND BY CALENDAR TYPE ===
Weekday Average: 451.45 missions/day
Weekend Average: 589.35 missions/day (Ratio: 1.31)

=== SYSTEM HYPOTHESIS ANSWER KEY MATRIX ===


,pattern,measured value,expected rule,status
0,Version Interventions Check,"v2.1=38.6, v2.2=36.5, v2.3=60.1",v2.2 < v2.1 < v2.3,PASS
1,Rain/Clear Risk Ratio,1.65,Ratio between 1.3 and 1.9,PASS
2,Old Town / Other Zones Ratio,2.01,Ratio between 1.6 and 2.5,PASS
3,Night / Day Risk Ratio,1.08,Ratio between 1.1 and 1.5,FAIL
4,Last-30-Days v2.3 vs v2.2 Ratio,1.76,Ratio above 1.3,PASS
5,Version v2.3 Dominant Issue,crosswalk_confusion,crosswalk_confusion,PASS
6,Rain SLA Penalty Points,41.02 pts,At least 5 points reduction,PASS
7,Old Town SLA Gap,"Lowest=Old Town, Gap=53.07 pts","Old Town is lowest, at least 10 pts below best",PASS
8,Rush Hour SLA Reduction,"Rush=52.11%, Normal=51.48%",Rush Hour OTP < Normal Hour OTP,FAIL
9,Tired Battery Robot Drain Signature,"['R023', 'R031', 'R007', 'R044', 'R019']","['R007', 'R019', 'R023', 'R031', 'R044']",PASS
